# 문화 중분류 후보 생성 및 기존 가맹점 중복 제외

이 노트북은 같은 폴더에 있는 두 원본 파일만 사용합니다.

- 소상공인시장진흥공단 서울 상가(상권)정보 CSV
- 문화누리카드 서울 오프라인 가맹점 목록 XLSX

처리 순서는 문화 중분류 매핑 → 자동포함/수동검토 분리 → 공식 가맹점과 상호명·좌표·주소 중복 매칭 → 기존 가맹점 제외 → CSV 두 개 저장입니다.

원본 파일은 수정하지 않습니다. 결과 파일 중분류_자동포함.csv와 중분류_수동검토.csv는 같은 이름이 이미 있으면 덮어씁니다.

In [1]:
from pathlib import Path
from dataclasses import dataclass
from collections import Counter, defaultdict
import difflib
import json
import math
import re
import zipfile
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)

DEFAULT_DATA_DIR = Path(r'C:\Users\USER\OneDrive\바탕 화면\상권데이터')
DATA_DIR = Path.cwd()
if not list(DATA_DIR.glob('소상공인시장진흥공단_상가(상권)정보_서울_*.csv')):
    DATA_DIR = DEFAULT_DATA_DIR

raw_candidates = sorted(DATA_DIR.glob('소상공인시장진흥공단_상가(상권)정보_서울_*.csv'))
official_candidates = sorted(DATA_DIR.glob('문화누리카드_서울_오프라인 가맹점 목록_*.xlsx'))
if not raw_candidates:
    raise FileNotFoundError(f'서울 상가정보 CSV를 찾지 못했습니다: {DATA_DIR}')
if not official_candidates:
    raise FileNotFoundError(f'문화누리카드 공식 가맹점 XLSX를 찾지 못했습니다: {DATA_DIR}')

RAW_CSV = raw_candidates[-1]
OFFICIAL_XLSX = official_candidates[-1]
AUTO_OUTPUT = DATA_DIR / '중분류_자동포함.csv'
MANUAL_OUTPUT = DATA_DIR / '중분류_수동검토.csv'
CHUNKSIZE = 100_000
SAVE_EXCLUSION_AUDIT = False

PROJECT_CATEGORIES = [
    '도서', '문화체험', '음악', '영상', '체육시설',
    '체육용품', '미술', '공연', '스포츠관람', '관광지',
]

SOURCE_COLUMNS = [
    '상가업소번호', '상호명', '지점명',
    '상권업종대분류코드', '상권업종대분류명',
    '상권업종중분류코드', '상권업종중분류명',
    '상권업종소분류코드', '상권업종소분류명',
    '표준산업분류코드', '표준산업분류명',
    '시도코드', '시도명', '시군구코드', '시군구명',
    '행정동코드', '행정동명', '법정동코드', '법정동명',
    '지번주소', '도로명주소', '신우편번호', '경도', '위도',
]

OUTPUT_BASE_COLUMNS = [
    '상가업소번호', '상호명',
    '상권업종대분류코드', '상권업종대분류명',
    '상권업종중분류코드', '상권업종중분류명',
    '상권업종소분류코드', '상권업종소분류명',
    '표준산업분류코드', '표준산업분류명',
    '시도명', '시군구명', '행정동명', '법정동명',
    '도로명주소', '지번주소', '경도', '위도',
]

MAPPING_COLUMNS = [
    'target_category', 'target_subcategory', 'include_yn',
    'culture_relevance_score', 'mapping_method', 'mapping_reason',
    'review_status', 'matched_keyword',
]

RENAME_COLUMNS = {
    'target_category': '문화중분류',
    'target_subcategory': '문화세부분류',
    'include_yn': '자동포함여부',
    'culture_relevance_score': '문화관련성점수',
    'mapping_method': '매핑방법',
    'mapping_reason': '매핑사유',
    'review_status': '검토상태',
    'matched_keyword': '매칭키워드',
}

print('상가정보:', RAW_CSV.name)
print('공식 가맹점:', OFFICIAL_XLSX.name)
print('결과 폴더:', DATA_DIR)

상가정보: 소상공인시장진흥공단_상가(상권)정보_서울_202606.csv
공식 가맹점: 문화누리카드_서울_오프라인 가맹점 목록_20260706.xlsx
결과 폴더: c:\Users\USER\OneDrive\바탕 화면\상권데이터


## 1. 문화 중분류 매핑 규칙

점수 3은 자동포함, 점수 1~2는 수동검토입니다. 여행은 현재 프로젝트 10개 중분류에 포함하지 않습니다.

In [2]:
SMALL_CODE_RULES = {
    'G21301': ('도서', '서점', 3, '상권업종 소분류가 서점'),
    'N11003': ('도서', '만화방', 3, '상권업종 소분류가 만화방'),
    'G21203': ('음악', '악기점', 3, '상권업종 소분류가 악기 소매업'),
    'G21303': ('음악', '음반점', 3, '상권업종 소분류가 음반·비디오물 소매업'),
    'P10609': ('음악', '음악교육', 3, '상권업종 소분류가 음악학원'),
    'G21801': ('미술', '미술품점', 3, '상권업종 소분류가 예술품 소매업'),
    'P10611': ('미술', '미술교육', 3, '상권업종 소분류가 미술학원'),
    'R10402': ('영상', '비디오감상', 3, '상권업종 소분류가 비디오방'),
    'P10601': ('체육시설', '무술교육', 3, '상권업종 소분류가 태권도·무술학원'),
    'P10603': ('체육시설', '요가·필라테스', 3, '상권업종 소분류가 요가·필라테스 학원'),
    'R10306': ('체육시설', '종합스포츠', 3, '상권업종 소분류가 종합 스포츠시설'),
    'R10307': ('체육시설', '체력단련', 3, '상권업종 소분류가 헬스장'),
    'R10308': ('체육시설', '수영장', 3, '상권업종 소분류가 수영장'),
    'R10309': ('체육시설', '볼링장', 3, '상권업종 소분류가 볼링장'),
    'R10310': ('체육시설', '당구장', 3, '상권업종 소분류가 당구장'),
    'R10311': ('체육시설', '골프연습장', 3, '상권업종 소분류가 골프 연습장'),
    'R10312': ('체육시설', '테니스장', 3, '상권업종 소분류가 테니스장'),
    'R10313': ('체육시설', '탁구장', 3, '상권업종 소분류가 탁구장'),
    'R10314': ('체육시설', '기타스포츠시설', 3, '상권업종 소분류가 기타 스포츠시설 운영업'),
    'R10316': ('체육시설', '라켓스포츠', 3, '상권업종 소분류가 스쿼시·라켓볼장'),
    'R10408': ('체육시설', '낚시터', 3, '상권업종 소분류가 낚시터 운영업'),
    'R10409': ('체육시설', '수상레저', 3, '상권업종 소분류가 수상·해양 레저업'),
    'G21304': ('체육용품', '운동용품점', 3, '상권업종 소분류가 운동용품 소매업'),
    'G21305': ('체육용품', '자전거점', 3, '상권업종 소분류가 자전거 소매업'),
    'N10501': ('여행', '여행사', 3, '상권업종 소분류가 여행사'),
    'G21302': ('미술', '문구·회화용품', 2, '문구와 회화용품이 혼합된 업종이므로 실제 취급품목 확인 필요'),
    'G21802': ('관광지', '기념품점', 2, '기념품점이나 관광시설·문화상품 여부 확인 필요'),
    'M11301': ('문화체험', '사진', 2, '사진촬영업이나 증명·상업촬영 등 용도 확인 필요'),
    'N10599': ('여행', '여행보조', 2, '여행 보조·예약 서비스의 대면 이용시설 여부 확인 필요'),
    'N11001': ('체육용품', '스포츠용품대여', 2, '스포츠·레크리에이션 용품 대여 업종'),
    'P10605': ('문화체험', '레크리에이션교육', 2, '체험·여가교육의 구체적 프로그램 확인 필요'),
    'P10607': ('문화체험', '청소년수련', 2, '청소년 수련시설의 이용 가능 프로그램 확인 필요'),
    'P10613': ('문화체험', '예술·스포츠교육', 2, '예술과 스포츠가 혼합된 소분류이므로 세부 서비스 확인 필요'),
    'R10414': ('문화체험', '보드게임', 2, '바둑·장기·체스 시설의 문화체험 활용성 확인 필요'),
}

@dataclass(frozen=True)
class CategoryRule:
    target_category: str
    target_subcategory: str
    score: int
    pattern: str
    reason: str

CATEGORY_RULES = [
    CategoryRule('영상', '영화관', 3, r'영화관 운영업|영화관$', '영화 상영 시설 업종'),
    CategoryRule('영상', '비디오감상', 3, r'비디오물 감상실 운영업', '영상 감상 시설 업종'),
    CategoryRule('도서', '서점', 3, r'서적 소매업', '서적 소매 업종'),
    CategoryRule('도서', '중고서점', 3, r'중고 서적(?: 및 음반)? 판매업', '중고 서적 판매 업종'),
    CategoryRule('음악', '악기점', 3, r'악기 소매업', '악기 소매 업종'),
    CategoryRule('음악', '음반점', 3, r'음반 및 비디오물 소매업', '음반·영상물 소매 업종'),
    CategoryRule('음악', '음악교육', 3, r'음악 학원|음악학원|실용음악', '음악 교육 업종'),
    CategoryRule('미술', '미술품점', 3, r'예술품 및 골동품 소매업', '예술품·골동품 소매 업종'),
    CategoryRule('미술', '미술교육', 3, r'미술 학원|미술학원', '미술 교육 업종'),
    CategoryRule('미술', '화랑', 3, r'화랑 운영업|화랑$', '화랑 운영 업종'),
    CategoryRule('공연', '공연장', 3, r'공연시설 운영업|공연장 운영업', '공연시설 운영 업종'),
    CategoryRule('공연', '공연기획', 3, r'공연 기획업|공연기획업', '공연 기획 업종'),
    CategoryRule('공연', '공연단체', 3, r'연극단체|무용 및 음악단체|공연 예술가', '공연예술 단체·예술가 업종'),
    CategoryRule('관광지', '박물관', 3, r'박물관 운영업|박물관$', '박물관 운영 업종'),
    CategoryRule('관광지', '자연·역사관광', 3, r'사적지 관리 운영업|식물원 및 동물원 운영업|자연공원 운영업', '자연·역사 관광시설 업종'),
    CategoryRule('관광지', '테마파크', 3, r'유원지 및 테마파크 운영업|테마파크', '유원지·테마파크 업종'),
    CategoryRule('여행', '여행사', 3, r'여행사업|여행사 및 기타 여행보조 서비스업', '여행사 업종'),
    CategoryRule('스포츠관람', '경기장', 3, r'스포츠 경기장 운영업|경기장 운영업', '스포츠 경기장 운영 업종'),
    CategoryRule('스포츠관람', '프로스포츠', 3, r'프로 스포츠 구단', '프로 스포츠 관람 관련 업종'),
    CategoryRule('체육시설', '체력단련', 3, r'체력 단련시설 운영업|체력단련시설 운영업', '체력단련시설 운영 업종'),
    CategoryRule('체육시설', '수영장', 3, r'수영장 운영업', '수영장 운영 업종'),
    CategoryRule('체육시설', '골프연습장', 3, r'골프연습장 운영업', '골프연습장 운영 업종'),
    CategoryRule('체육시설', '생활체육', 3, r'볼링장 운영업|당구장 운영업|기타 스포츠시설 운영업', '생활체육시설 운영 업종'),
    CategoryRule('체육시설', '무술교육', 3, r'태권도 및 무술 교육기관', '태권도·무술 교육 업종'),
    CategoryRule('체육시설', '스포츠교육', 3, r'스포츠 교육기관', '스포츠 교육 업종'),
    CategoryRule('체육용품', '운동용품점', 3, r'운동 및 경기용품 소매업|운동용품 소매업', '운동·경기용품 소매 업종'),
    CategoryRule('체육용품', '자전거점', 3, r'자전거 및 기타 운송장비 소매업', '자전거 소매 업종'),
    CategoryRule('체육용품', '레저용품점', 3, r'낚시 및 수렵용구 소매업|캠핑용품 소매업', '낚시·캠핑 등 레저용품 소매 업종'),
    CategoryRule('문화체험', '예술교육', 3, r'기타 예술학원|기타 예술 교육기관|기타 예술교육 서비스업', '예술교육 업종'),
    CategoryRule('문화체험', '공예', 2, r'공예품 소매업|도자기.*제조업|공예.*제조업', '공예 체험 가능성이 있으나 단순 제조·소매일 수 있음'),
    CategoryRule('문화체험', '사진', 2, r'사진 촬영 및 처리업|인물사진 및 행사용 영상 촬영업', '문화활동 관련성이 있으나 증명·상업촬영일 수 있음'),
    CategoryRule('문화체험', '레크리에이션교육', 2, r'레크리에이션 교육기관', '체험·여가교육 가능성이 있으나 서비스 확인 필요'),
    CategoryRule('공연', '창작예술', 2, r'창작 및 예술관련 서비스업', '공연·창작 관련성이 있으나 실제 시설 여부 확인 필요'),
    CategoryRule('영상', '영상제작', 2, r'영화, 비디오물 및 방송프로그램 제작업|영상물 제작업', '영상 관련 업종이나 이용자 방문시설 여부 확인 필요'),
]

KEYWORD_RULES = [
    ('도서', '서점·북카페', r'서점|북카페|북스|책방'),
    ('음악', '악기·음악공간', r'악기|피아노|바이올린|첼로|드럼|음반|실용음악|음악연습'),
    ('영상', '영화·영상공간', r'영화|시네마|비디오|영상'),
    ('미술', '미술·전시공간', r'미술|갤러리|화랑|아틀리에|도예|캘리그라피'),
    ('공연', '공연·무대공간', r'공연|극장|연극|무용|발레|콘서트'),
    ('문화체험', '복합문화·체험', r'문화센터|문화공간|복합문화|체험공간|공방|공예|클래스'),
    ('체육시설', '운동시설', r'체육관|스포츠센터|헬스|피트니스|수영|골프연습|태권도|요가|필라테스|클라이밍'),
    ('체육용품', '스포츠·레저용품', r'운동용품|스포츠용품|아웃도어|자전거|캠핑|낚시'),
    ('관광지', '관광시설', r'박물관|미술관|전시관|테마파크|수목원|동물원'),
    ('여행', '여행서비스', r'여행사|트래블|투어'),
    ('스포츠관람', '경기장', r'경기장|스타디움'),
]

print('소분류코드 규칙:', len(SMALL_CODE_RULES))
print('표준산업분류 보완 규칙:', len(CATEGORY_RULES))
print('상호명 키워드 규칙:', len(KEYWORD_RULES))

소분류코드 규칙: 34
표준산업분류 보완 규칙: 34
상호명 키워드 규칙: 11


## 2. 원본 상가정보 전수 매핑

55만 행을 10만 행씩 읽어 문화 관련 후보만 메모리에 보관합니다.

In [3]:
def classify_chunk(chunk):
    out = chunk.copy()
    out['target_category'] = ''
    out['target_subcategory'] = ''
    out['include_yn'] = 'N'
    out['culture_relevance_score'] = 0
    out['mapping_method'] = '업종분류'
    out['mapping_reason'] = '문화 관련 업종 규칙에 해당하지 않음'
    out['review_status'] = '자동제외'
    out['matched_keyword'] = ''

    small_codes = out['상권업종소분류코드'].fillna('').astype(str)
    for code_value, (category, subcategory, score, reason) in SMALL_CODE_RULES.items():
        mask = out['target_category'].eq('') & small_codes.eq(code_value)
        if not mask.any():
            continue
        out.loc[mask, 'target_category'] = category
        out.loc[mask, 'target_subcategory'] = subcategory
        out.loc[mask, 'include_yn'] = 'Y' if score == 3 else '검토'
        out.loc[mask, 'culture_relevance_score'] = score
        out.loc[mask, 'mapping_method'] = '상권업종 소분류코드'
        out.loc[mask, 'mapping_reason'] = reason
        out.loc[mask, 'review_status'] = '자동분류' if score == 3 else '수동검토 필요'

    category_names = (
        out['상권업종소분류명'].fillna('').astype(str)
        + ' | ' + out['표준산업분류명'].fillna('').astype(str)
    )
    for rule in CATEGORY_RULES:
        mask = out['target_category'].eq('') & category_names.str.contains(
            rule.pattern, case=False, regex=True, na=False
        )
        if not mask.any():
            continue
        effective_score = min(rule.score, 2)
        out.loc[mask, 'target_category'] = rule.target_category
        out.loc[mask, 'target_subcategory'] = rule.target_subcategory
        out.loc[mask, 'include_yn'] = '검토'
        out.loc[mask, 'culture_relevance_score'] = effective_score
        out.loc[mask, 'mapping_method'] = '표준산업분류 보완'
        out.loc[mask, 'mapping_reason'] = rule.reason + '; 상권업종 소분류 불일치 가능성으로 자동 포함하지 않음'
        out.loc[mask, 'review_status'] = '수동검토 필요'

    normalized_names = out['상호명'].fillna('').astype(str).str.replace(r'\s+', '', regex=True)
    for category, subcategory, pattern in KEYWORD_RULES:
        mask = out['target_category'].eq('') & normalized_names.str.contains(
            pattern, case=False, regex=True, na=False
        )
        if not mask.any():
            continue
        out.loc[mask, 'target_category'] = category
        out.loc[mask, 'target_subcategory'] = subcategory
        out.loc[mask, 'include_yn'] = '검토'
        out.loc[mask, 'culture_relevance_score'] = 1
        out.loc[mask, 'mapping_method'] = '상호명 키워드'
        out.loc[mask, 'mapping_reason'] = '업종분류로 포함되지 않았으나 상호명에서 문화 관련 키워드 발견'
        out.loc[mask, 'review_status'] = '수동검토 필요'
        out.loc[mask, 'matched_keyword'] = normalized_names.loc[mask].str.extract(
            f'({pattern})', expand=False
        )
    return out

header = pd.read_csv(RAW_CSV, encoding='utf-8-sig', nrows=0)
missing_source_columns = sorted(set(SOURCE_COLUMNS) - set(header.columns))
if missing_source_columns:
    raise ValueError(f'원본 필수 컬럼 누락: {missing_source_columns}')

auto_parts = []
manual_parts = []
source_row_count = 0

for chunk_no, chunk in enumerate(pd.read_csv(
    RAW_CSV, encoding='utf-8-sig', usecols=SOURCE_COLUMNS,
    dtype={'상가업소번호': str}, chunksize=CHUNKSIZE, low_memory=False,
), start=1):
    source_row_count += len(chunk)
    mapped = classify_chunk(chunk)
    project = mapped['target_category'].isin(PROJECT_CATEGORIES)
    auto_parts.append(mapped.loc[project & mapped['culture_relevance_score'].eq(3)])
    manual_parts.append(mapped.loc[project & mapped['culture_relevance_score'].isin([1, 2])])
    print(f'청크 {chunk_no}: 누적 {source_row_count:,}행', end='\r')

auto_candidates = pd.concat(auto_parts, ignore_index=True)
manual_candidates = pd.concat(manual_parts, ignore_index=True)

mapping_counts = pd.DataFrame([
    ('원본', source_row_count),
    ('자동포함 후보', len(auto_candidates)),
    ('수동검토 후보', len(manual_candidates)),
], columns=['구분', '행수'])
display(mapping_counts)

EXPECTED_MAPPING = {'원본': 554_092, '자동포함 후보': 28_591, '수동검토 후보': 16_054}
for label, expected in EXPECTED_MAPPING.items():
    actual = int(mapping_counts.loc[mapping_counts['구분'].eq(label), '행수'].iloc[0])
    if actual != expected:
        print(f'주의: {label} 현재값 {actual:,} / 202606 기준값 {expected:,}')

,구분,행수
0,원본,554092
1,자동포함 후보,28591
2,수동검토 후보,16054


## 3. 공식 문화누리카드 가맹점 불러오기

pandas의 Excel 엔진이 없으면 파이썬 표준 라이브러리로 첫 번째 시트를 읽는 대체 경로를 사용합니다.

In [4]:
def excel_column_index(cell_reference):
    letters = ''.join(character for character in cell_reference if character.isalpha())
    value = 0
    for character in letters:
        value = value * 26 + (ord(character.upper()) - ord('A') + 1)
    return value - 1

def read_first_sheet_without_openpyxl(path):
    ns = {'x': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
    with zipfile.ZipFile(path) as archive:
        shared_strings = []
        if 'xl/sharedStrings.xml' in archive.namelist():
            root = ET.fromstring(archive.read('xl/sharedStrings.xml'))
            for item in root.findall('x:si', ns):
                shared_strings.append(''.join(node.text or '' for node in item.findall('.//x:t', ns)))
        sheet_names = sorted(
            name for name in archive.namelist()
            if name.startswith('xl/worksheets/sheet') and name.endswith('.xml')
        )
        root = ET.fromstring(archive.read(sheet_names[0]))
        rows = []
        max_column = 0
        for row_node in root.findall('.//x:sheetData/x:row', ns):
            values = {}
            for cell in row_node.findall('x:c', ns):
                column = excel_column_index(cell.attrib.get('r', 'A1'))
                max_column = max(max_column, column)
                cell_type = cell.attrib.get('t')
                value_node = cell.find('x:v', ns)
                if cell_type == 'inlineStr':
                    value = ''.join(node.text or '' for node in cell.findall('.//x:t', ns))
                elif value_node is None:
                    value = None
                elif cell_type == 's':
                    value = shared_strings[int(value_node.text)]
                elif cell_type in {'str', 'b'}:
                    value = value_node.text
                else:
                    text = value_node.text
                    try:
                        value = float(text)
                    except (TypeError, ValueError):
                        value = text
                values[column] = value
            rows.append(values)
    return pd.DataFrame([
        [row.get(column) for column in range(max_column + 1)] for row in rows
    ])

def read_official_raw(path):
    try:
        return pd.read_excel(path, header=None)
    except (ImportError, ModuleNotFoundError):
        print('openpyxl이 없어 표준 라이브러리 방식으로 XLSX를 읽습니다.')
        return read_first_sheet_without_openpyxl(path)

official_raw = read_official_raw(OFFICIAL_XLSX)
if official_raw.shape[0] < 3 or official_raw.shape[1] < 14:
    raise ValueError(f'공식 가맹점 표 구조를 인식하지 못했습니다: {official_raw.shape}')

official = pd.DataFrame({
    'official_name': official_raw.iloc[2:, 1],
    'official_large_category': official_raw.iloc[2:, 3],
    'official_middle_category': official_raw.iloc[2:, 4],
    'official_small_category': official_raw.iloc[2:, 5],
    'official_lat': official_raw.iloc[2:, 6],
    'official_lon': official_raw.iloc[2:, 7],
    'official_district': official_raw.iloc[2:, 12],
    'official_address': official_raw.iloc[2:, 13],
}).reset_index(drop=True)

official['official_lat'] = pd.to_numeric(official['official_lat'], errors='coerce')
official['official_lon'] = pd.to_numeric(official['official_lon'], errors='coerce')
official = official.loc[official['official_name'].notna()].copy().reset_index(drop=True)
official['coordinate_valid'] = (
    official['official_lat'].between(33, 39)
    & official['official_lon'].between(124, 132)
)

print('공식 가맹점:', f'{len(official):,}개')
print('공식 가맹점 좌표 오류:', f'{(~official["coordinate_valid"]).sum():,}개')
display(official.head(3))

공식 가맹점: 4,727개
공식 가맹점 좌표 오류: 13개


,official_name,official_large_category,official_middle_category,official_small_category,official_lat,official_lon,official_district,official_address,coordinate_valid
0,피트니스써밋,체육,체육시설,체육시설,37.585338,126.914969,서대문구,서울특별시 서대문구 응암로 145 3층,True
1,상지악기,문화,음악,음악,37.535248,127.132664,강동구,서울특별시 강동구 천호대로 1092 에스케이 허브진 1층 114-1호 (주)상지악기,True
2,스파이럴치어리딩학원(스파이럴 치어리딩학원),체육,체육시설,체육시설,37.541578,127.094914,광진구,서울특별시 광진구 광나루로 604 6층,True


## 4. 기존 가맹점 중복 매칭

공통 ID가 없으므로 정규화 상호명, 이름 유사도, 좌표 거리, 주소를 함께 사용합니다. 좌표가 잘못된 공식 가맹점은 동일 자치구의 상호명·주소로 제한 보완합니다.

In [5]:
def normalize_name(value):
    if pd.isna(value):
        return ''
    text = str(value).lower()
    text = re.sub(r'\([^)]*(?:주식회사|유한회사|사단법인|재단법인|지점|점)\)', '', text)
    text = re.sub(r'(?:주식회사|유한회사|유한책임회사|사단법인|재단법인|협동조합)', '', text)
    text = re.sub(r'(?:서울특별시|서울시)$', '', text)
    return re.sub(r'[^0-9a-z가-힣]', '', text)

def normalize_address(value):
    if pd.isna(value):
        return ''
    text = str(value).lower().replace('서울특별시', '').replace('서울시', '')
    return re.sub(r'[^0-9a-z가-힣]', '', text)

def name_score(left, right):
    if not left or not right:
        return 0.0
    return difflib.SequenceMatcher(None, left, right).ratio()

def haversine_m(lon1, lat1, lon2, lat2):
    radius = 6_371_008.8
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    value = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * radius * math.asin(math.sqrt(value))

official['name_norm'] = official['official_name'].map(normalize_name)
official['address_norm'] = official['official_address'].map(normalize_address)
official['cell_x'] = np.nan
official['cell_y'] = np.nan
valid_coordinates = official['coordinate_valid']
official.loc[valid_coordinates, 'cell_x'] = np.floor(official.loc[valid_coordinates, 'official_lon'] * 1000)
official.loc[valid_coordinates, 'cell_y'] = np.floor(official.loc[valid_coordinates, 'official_lat'] * 1000)

official_buckets = defaultdict(list)
for index, row in official.loc[valid_coordinates, ['cell_x', 'cell_y']].iterrows():
    official_buckets[(int(row['cell_x']), int(row['cell_y']))].append(int(index))
invalid_coordinate_indices = official.index[~valid_coordinates].astype(int).tolist()

def match_existing_merchant(row):
    lon = pd.to_numeric(pd.Series([row.get('경도')]), errors='coerce').iloc[0]
    lat = pd.to_numeric(pd.Series([row.get('위도')]), errors='coerce').iloc[0]
    if not np.isfinite(lon) or not np.isfinite(lat):
        return None

    base_name = normalize_name(row.get('상호명'))
    branch_name = normalize_name(row.get('지점명'))
    variants = {base_name}
    if branch_name:
        variants.add(base_name + branch_name)
    variants.discard('')
    if not variants:
        return None
    address_norm = normalize_address(row.get('도로명주소'))

    cell_x = int(math.floor(float(lon) * 1000))
    cell_y = int(math.floor(float(lat) * 1000))
    nearby = []
    for dx in range(-3, 4):
        for dy in range(-3, 4):
            nearby.extend(official_buckets.get((cell_x + dx, cell_y + dy), []))

    best = None
    for official_index in nearby:
        target = official.iloc[official_index]
        distance = haversine_m(float(lon), float(lat), float(target['official_lon']), float(target['official_lat']))
        if distance > 250:
            continue
        target_name = str(target['name_norm'])
        similarity = max(name_score(source, target_name) for source in variants)
        exact_name = target_name in variants and len(target_name) >= 3
        contained = any(
            min(len(source), len(target_name)) >= 4 and (source in target_name or target_name in source)
            for source in variants
        )
        contained_lengths = [
            min(len(source), len(target_name)) for source in variants
            if source in target_name or target_name in source
        ]
        shorter_length = max(contained_lengths, default=0)
        address_similarity = name_score(address_norm, str(target['address_norm']))
        category_same = str(row.get('target_category', '')) == str(target['official_middle_category'])

        rule = ''
        if exact_name and distance <= 100:
            rule = '정규화상호명일치+100m이내'
        elif similarity >= 0.90 and distance <= 50:
            rule = '상호명유사도0.90이상+50m이내'
        elif contained and shorter_length >= 5 and distance <= 50:
            rule = '상호명포함관계5자이상+50m이내'
        elif contained and similarity >= 0.75 and distance <= 60:
            rule = '상호명포함관계+유사도0.75이상+60m이내'
        elif similarity >= 0.75 and distance <= 20:
            rule = '상호명유사도0.75이상+20m이내'
        elif category_same and similarity >= 0.70 and address_similarity >= 0.95 and distance <= 15:
            rule = '동일중분류+주소유사도0.95이상+15m이내'
        if not rule:
            continue

        rank_key = (similarity, address_similarity, -distance)
        if best is None or rank_key > best['rank_key']:
            best = {
                'rank_key': rank_key,
                'official_name': target['official_name'],
                'official_middle_category': target['official_middle_category'],
                'distance_m': distance,
                'name_similarity': similarity,
                'address_similarity': address_similarity,
                'match_rule': rule,
            }
    if best is not None:
        return best

    candidate_district = str(row.get('시군구명', '')).strip()
    for official_index in invalid_coordinate_indices:
        target = official.iloc[official_index]
        if candidate_district != str(target['official_district']).strip():
            continue
        target_name = str(target['name_norm'])
        similarity = max(name_score(source, target_name) for source in variants)
        contained = any(
            min(len(source), len(target_name)) >= 5 and (source in target_name or target_name in source)
            for source in variants
        )
        address_similarity = name_score(address_norm, str(target['address_norm']))
        exact_name = target_name in variants and len(target_name) >= 3
        if exact_name and address_similarity >= 0.60:
            rule = '공식좌표오류+상호명일치+주소보완'
        elif contained and address_similarity >= 0.75:
            rule = '공식좌표오류+상호명포함+주소보완'
        elif similarity >= 0.90 and address_similarity >= 0.85:
            rule = '공식좌표오류+상호주소고유사'
        else:
            continue
        return {
            'rank_key': (similarity, address_similarity, 0.0),
            'official_name': target['official_name'],
            'official_middle_category': target['official_middle_category'],
            'distance_m': np.nan,
            'name_similarity': similarity,
            'address_similarity': address_similarity,
            'match_rule': rule,
        }
    return None

def exclude_existing_merchants(candidates, label):
    matches = []
    for position, (_, row) in enumerate(candidates.iterrows(), start=1):
        matches.append(match_existing_merchant(row))
        if position % 5_000 == 0:
            print(f'{label}: {position:,}/{len(candidates):,} 매칭', end='\r')
    matched = pd.Series([value is not None for value in matches], index=candidates.index)
    remaining = candidates.loc[~matched].copy()
    excluded = candidates.loc[matched].copy()
    if not excluded.empty:
        details = [value for value in matches if value is not None]
        excluded['공식가맹점명'] = [value['official_name'] for value in details]
        excluded['공식중분류'] = [value['official_middle_category'] for value in details]
        excluded['매칭거리m'] = [value['distance_m'] for value in details]
        excluded['상호명유사도'] = [value['name_similarity'] for value in details]
        excluded['주소유사도'] = [value['address_similarity'] for value in details]
        excluded['중복판정규칙'] = [value['match_rule'] for value in details]
    print(f'{label}: 기존 가맹점 {matched.sum():,}개 제외')
    return remaining, excluded

auto_remaining, auto_excluded = exclude_existing_merchants(auto_candidates, '자동포함')
manual_remaining, manual_excluded = exclude_existing_merchants(manual_candidates, '수동검토')

자동포함: 기존 가맹점 1,334개 제외
수동검토: 기존 가맹점 559개 제외매칭


## 5. CSV 저장 및 검증

요청한 18개 원본 컬럼과 문화 중분류·매핑 근거 8개 컬럼을 저장합니다.

In [6]:
output_columns = OUTPUT_BASE_COLUMNS + MAPPING_COLUMNS
missing_output_columns = sorted(set(output_columns) - set(auto_remaining.columns))
if missing_output_columns:
    raise ValueError(f'출력 필수 컬럼 누락: {missing_output_columns}')

auto_output_data = auto_remaining[output_columns].rename(columns=RENAME_COLUMNS)
manual_output_data = manual_remaining[output_columns].rename(columns=RENAME_COLUMNS)

auto_output_data.to_csv(AUTO_OUTPUT, index=False, encoding='utf-8-sig')
manual_output_data.to_csv(MANUAL_OUTPUT, index=False, encoding='utf-8-sig')

if SAVE_EXCLUSION_AUDIT:
    auto_excluded.to_csv(DATA_DIR / '_검증용_자동포함_제외목록.csv', index=False, encoding='utf-8-sig')
    manual_excluded.to_csv(DATA_DIR / '_검증용_수동검토_제외목록.csv', index=False, encoding='utf-8-sig')

cross_duplicates = len(
    set(auto_output_data['상가업소번호']) & set(manual_output_data['상가업소번호'])
)

validation = pd.DataFrame([
    ('공식 가맹점', len(official), '', ''),
    ('자동포함 매핑 후보', len(auto_candidates), len(auto_excluded), len(auto_output_data)),
    ('수동검토 매핑 후보', len(manual_candidates), len(manual_excluded), len(manual_output_data)),
], columns=['구분', '입력행수', '기존가맹점제외', '최종행수'])
display(validation)

file_validation = pd.DataFrame([
    ('중분류_자동포함.csv', len(auto_output_data), len(auto_output_data.columns), auto_output_data['상가업소번호'].duplicated().sum()),
    ('중분류_수동검토.csv', len(manual_output_data), len(manual_output_data.columns), manual_output_data['상가업소번호'].duplicated().sum()),
], columns=['파일', '행수', '컬럼수', '상가업소번호중복'])
display(file_validation)

EXPECTED_FINAL = {
    '중분류_자동포함.csv': 27_257,
    '중분류_수동검토.csv': 15_495,
}
for _, row in file_validation.iterrows():
    expected = EXPECTED_FINAL[row['파일']]
    if row['행수'] != expected:
        print(f"주의: {row['파일']} 현재 {row['행수']:,}행 / 202606·20260706 기준 {expected:,}행")

assert auto_output_data['상가업소번호'].is_unique
assert manual_output_data['상가업소번호'].is_unique
assert cross_duplicates == 0
assert set(auto_output_data['문화중분류']).issubset(PROJECT_CATEGORIES)
assert set(manual_output_data['문화중분류']).issubset(PROJECT_CATEGORIES)
assert AUTO_OUTPUT.read_bytes()[:3] == b'\xef\xbb\xbf'
assert MANUAL_OUTPUT.read_bytes()[:3] == b'\xef\xbb\xbf'

print('파일 간 상가업소번호 중복:', cross_duplicates)
print('저장 완료:', AUTO_OUTPUT)
print('저장 완료:', MANUAL_OUTPUT)

,구분,입력행수,기존가맹점제외,최종행수
0,공식 가맹점,4727,,
1,자동포함 매핑 후보,28591,1334,27257
2,수동검토 매핑 후보,16054,559,15495


,파일,행수,컬럼수,상가업소번호중복
0,중분류_자동포함.csv,27257,26,0
1,중분류_수동검토.csv,15495,26,0


파일 간 상가업소번호 중복: 0
저장 완료: c:\Users\USER\OneDrive\바탕 화면\상권데이터\중분류_자동포함.csv
저장 완료: c:\Users\USER\OneDrive\바탕 화면\상권데이터\중분류_수동검토.csv


## 결과 해석

- 자동포함: 업종 소분류코드가 문화시설과 직접 연결되고 기존 공식 가맹점과 겹치지 않는 후보
- 수동검토: 혼합 업종, 표준산업분류 보완, 상호명 키워드 후보 중 기존 공식 가맹점과 겹치지 않는 시설
- 공식 가맹점과 후보 데이터에 공통 ID가 없으므로 중복 판정은 상호명·좌표·주소를 결합한 보수적 레코드 매칭 결과입니다.